# OUR REQUIREMENT
Design and Implement an end-to-end Sales Analytics pipeline using the TPCH dataset by following Medallion Architecture  

### BRONZE LAYER -
Ingesting raw CSV data from the source volume using Auto Loader and storing the data in Delta tables as-is.The Bronze layer preserves the raw structure without applying schema enforcement, while schema definition and transformations are handled later in the Silver layer

In [0]:
# Defining Catalog, Schema and Volume Path
catalog = "az_adb_simbus_training"
schema = "adarsh_training"
volume_path = "/Volumes/az_adb_simbus_training/adarsh_training/dataingestion"
# Using DBFS for checkpoints since checkpoints volume doesn't exist
checkpoint_path = "/dbfs/checkpoints/adarsh_training/dataingestion/"
# List of dataset file name patterns to be processed
datasets = ["CustomersRaw_Data", "LineitemRaw_Data", "OrdersRaw_Data", "NationRaw_Data"]
# Using For Loop to loop through each dataset and create an ingestion pipeline
for table_name in datasets:
    # Read data using Databricks Auto Loader (cloudFiles)
    df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("header", "true")
    .option("pathGlobFilter", f"*{table_name}*")
    .load(volume_path)
    )
    # Write streaming data into a Delta table (Bronze layer) 
    (df.writeStream
    .option("checkpointLocation", f"{checkpoint_path}data_{table_name}")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.{schema}.bronze_{table_name}")
    )  
    print(f"Auto Loader: bronze_{table_name} table is created.")